# 🛰️ Notebook 3 — Master Evaluation, EDA & Publication Report
## Satellite Aerial Object Detection · 5-Model Comparison + 3-Slice SAHI

**What this notebook does:**
- Loads outputs from Notebook 1 (YOLO11n/s/m) and Notebook 2 (RT-DETR-L, YOLOv8m)
- Shows everything that was done in both notebooks in full detail
- Complete raw dataset EDA (10 figures)
- Sample images: train / val / test with ground-truth boxes
- Model predictions before vs after SAHI on all splits
- Full 20-metric comparison table, 15-param audit, 3-slice SAHI ablation
- 12 publication-ready figures (300 DPI) + LaTeX table

> **No GPU required.** Pure evaluation — CPU is fine.  
> **Update the two `INPUT_*` paths in Cell 1 before running.**

## 📦 Cell 1 — Setup · Load NB1 + NB2 Results · Verify Paths

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP + LOAD ALL NB1 + NB2 OUTPUTS
# ══════════════════════════════════════════════════════════════════════

!pip install ultralytics sahi seaborn scipy opencv-python-headless -q

import os, json, warnings, random, gc, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter
import torch

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "bold", "figure.dpi": 100})

def save_close(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    gc.collect()
    print(f"   ✅ {os.path.basename(path)}")

# ── Device ──────────────────────────────────────────────────────────
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ══════════════════════════════════════════════════════════════════════
# ⚠️  UPDATE THESE TWO PATHS TO YOUR KAGGLE DATASET NAMES
# ══════════════════════════════════════════════════════════════════════
NB1_DATASET  = "/kaggle/input/datasets/rifat4018/resultsyolo11"   # NB1 output
NB2_DATASET  = "/kaggle/input/datasets/rifat4018/resultsrfdetr"   # NB2 output
DATASET_PATH = "/kaggle/input/datasets/rifat4018/satelite-coco"   # raw COCO
WORK_DIR     = "/kaggle/working"
PLOT_DIR     = f"{WORK_DIR}/paper_figures"
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Paths assembled from actual file locations you described ─────────
NB1_CSV      = f"{NB1_DATASET}/saved_models/nb1_results.csv"
NB2_CSV      = f"{NB2_DATASET}/saved_models/nb2_results.csv"
EDA_CSV      = f"{NB1_DATASET}/saved_models/eda_summary.csv"

MODEL_PATHS = {
    "YOLO11n":   f"{NB1_DATASET}/saved_models/YOLO11n_best.pt",
    "YOLO11s":   f"{NB1_DATASET}/saved_models/YOLO11s_best.pt",
    "YOLO11m":   f"{NB1_DATASET}/saved_models/YOLO11m_best.pt",
    "RT-DETR-L": f"{NB2_DATASET}/saved_models/RT-DETR-L_best.pt",
    "YOLOv8m":   f"{NB2_DATASET}/saved_models/YOLOv8m_best.pt",
}
YAML_PATH    = f"{NB2_DATASET}/data.yaml"   # data.yaml saved in NB2 dataset

TRAIN_JSON   = f"{DATASET_PATH}/train/_annotations.coco.json"
VAL_JSON     = f"{DATASET_PATH}/valid/_annotations.coco.json"
TEST_JSON    = f"{DATASET_PATH}/test/_annotations.coco.json"
TRAIN_IMG    = f"{DATASET_PATH}/train"
VAL_IMG      = f"{DATASET_PATH}/valid"
TEST_IMG     = f"{DATASET_PATH}/test"

# ── Verify all paths ────────────────────────────────────────────────
print("\n" + "="*65)
print("PATH VERIFICATION")
print("="*65)
all_ok = True
for label, path in [
    ("NB1 results CSV",   NB1_CSV),
    ("NB2 results CSV",   NB2_CSV),
    ("EDA summary CSV",   EDA_CSV),
    ("data.yaml",         YAML_PATH),
    ("YOLO11n weights",   MODEL_PATHS["YOLO11n"]),
    ("YOLO11s weights",   MODEL_PATHS["YOLO11s"]),
    ("YOLO11m weights",   MODEL_PATHS["YOLO11m"]),
    ("RT-DETR-L weights", MODEL_PATHS["RT-DETR-L"]),
    ("YOLOv8m weights",   MODEL_PATHS["YOLOv8m"]),
    ("Train COCO JSON",   TRAIN_JSON),
    ("Val COCO JSON",     VAL_JSON),
    ("Test COCO JSON",    TEST_JSON),
]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌ MISSING'}  {label:<22} {path}")
    if not ok: all_ok = False

if not all_ok:
    print("\n⚠️  Some paths are missing — update NB1_DATASET / NB2_DATASET above.")
else:
    print("\n  ✅ All paths verified.")

# ── Load CSVs ───────────────────────────────────────────────────────
df_nb1 = pd.read_csv(NB1_CSV) if os.path.exists(NB1_CSV) else pd.DataFrame()
df_nb2 = pd.read_csv(NB2_CSV) if os.path.exists(NB2_CSV) else pd.DataFrame()
df     = pd.concat([df_nb1, df_nb2], ignore_index=True)

if "base_model" not in df.columns:
    df["base_model"] = df["Model"].str.replace(r"_S\d+$", "", regex=True)
if "sahi_slice" not in df.columns:
    df["sahi_slice"] = df["Model"].str.extract(r"_S(\d+)$").astype(float)

BASE_MODELS  = df["base_model"].unique().tolist() if not df.empty else []
SAHI_SIZES_R = sorted(df["sahi_slice"].unique().tolist(), reverse=True) if not df.empty else [480,320,160]
PALETTE      = ["#2196F3","#FF9800","#4CAF50","#E91E63","#9C27B0"]
PALETTE_D    = dict(zip(BASE_MODELS, PALETTE[:len(BASE_MODELS)]))

print(f"\n  NB1 rows: {len(df_nb1)}   NB2 rows: {len(df_nb2)}   Total: {len(df)}")
print(f"  Base models : {BASE_MODELS}")
print(f"  SAHI slices : {SAHI_SIZES_R}")

# ── Load COCO data ──────────────────────────────────────────────────
def load_coco(p):
    with open(p) as f: return json.load(f)

train_data  = load_coco(TRAIN_JSON)
val_data    = load_coco(VAL_JSON)
test_data   = load_coco(TEST_JSON)
cat_map     = {c["id"]: c["name"] for c in train_data["categories"]}
CLASS_NAMES = list(cat_map.values())
COLORS      = {"plane":"#2196F3","ship":"#FF9800","vehicle":"#4CAF50"}
CV_COLORS   = {"plane":(255,80,80),"ship":(80,200,80),"vehicle":(80,80,255)}

print(f"\n  Classes: {CLASS_NAMES}")
print("\n📌 Cell 1 done. Proceed to Cell 2 — Full Dataset EDA.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 26.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Device: cpu

PATH VERIFICATION
  ✅  NB1 results CSV        /kaggle/input/datasets/rifat4018/resultsyolo11/saved_models/nb1_results.csv
  ✅  NB2 results CSV        /kaggle/input/datasets/rifat4018/resultsrfdetr/saved_models/nb2_results.csv
  ✅  EDA summary CSV        /kaggle/input/datasets/rifat4018/resultsyolo11/saved_models/eda_summary.csv
  ✅  data.yaml              /kaggle/input/datasets/rifat401

## 🔬 Cell 2 — Complete Dataset EDA (10 Figures + All Tables)

In [9]:
print("CLASS_NAMES:", CLASS_NAMES)

all_classes = set(cat_map.values())
print("Dataset classes:", all_classes)

CLASS_NAMES: ['objects', 'plane', 'ship', 'vehicle']
Dataset classes: {'vehicle', 'plane', 'ship', 'objects'}


In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — COMPLETE RAW DATASET EDA
# ══════════════════════════════════════════════════════════════════════

splits    = {"Train": train_data, "Val": val_data, "Test": test_data}
split_img = {"Train": TRAIN_IMG,  "Val": VAL_IMG,  "Test": TEST_IMG}

# Pre-compute stats
widths  = [i["width"]  for i in train_data["images"]]
heights = [i["height"] for i in train_data["images"]]
unique_res = set(zip(widths, heights))
sizes  = np.array([a["bbox"][2]*a["bbox"][3] for a in train_data["annotations"]])
ratios = [a["bbox"][2]/a["bbox"][3] for a in train_data["annotations"] if a["bbox"][3]>0]
small  = int(np.sum(sizes <  32**2))
medium = int(np.sum((sizes>=32**2)&(sizes<96**2)))
large  = int(np.sum(sizes >= 96**2))
total  = len(sizes)
img_obj = Counter(a["image_id"] for a in train_data["annotations"])
for img in train_data["images"]:
    if img["id"] not in img_obj: img_obj[img["id"]] = 0
density = list(img_obj.values())
train_cnts = Counter(cat_map[a["category_id"]] for a in train_data["annotations"])
val_cnts   = Counter(cat_map[a["category_id"]] for a in val_data["annotations"])
test_cnts  = Counter(cat_map[a["category_id"]] for a in test_data["annotations"])

# ── Print full stats table ───────────────────────────────────────────
print("="*65)
print("COMPLETE RAW DATASET STATISTICS")
print("="*65)
rows_stat = []
for sname, d in splits.items():
    n_imgs  = len(d["images"])
    n_anns  = len(d["annotations"])
    n_empty = n_imgs - len(set(a["image_id"] for a in d["annotations"]))
    cnts    = Counter(cat_map[a["category_id"]] for a in d["annotations"])
    rows_stat.append({
        "Split":       sname,
        "Images":      n_imgs,
        "Annotations": n_anns,
        "Empty imgs":  n_empty,
        "Ann/img":     round(n_anns/max(n_imgs,1),2),
        "plane":       cnts.get("plane",0),
        "ship":        cnts.get("ship",0),
        "vehicle":     cnts.get("vehicle",0),
    })
print(pd.DataFrame(rows_stat).to_string(index=False))
print(f"\n  Image resolution : {np.mean(widths):.0f}×{np.mean(heights):.0f}  unique combos: {len(unique_res)}")
print(f"  Small (<32²): {small:,}({100*small/total:.1f}%)  "
      f"Med: {medium:,}({100*medium/total:.1f}%)  Large: {large:,}({100*large/total:.1f}%)")
print(f"  Aspect ratio : mean={np.mean(ratios):.3f}  std={np.std(ratios):.3f}")
print(f"  Ann density  : mean={np.mean(density):.2f}  max={max(density)}  std={np.std(density):.2f}")

neg_x  = sum(1 for a in train_data["annotations"] if a["bbox"][0]<0)
neg_y  = sum(1 for a in train_data["annotations"] if a["bbox"][1]<0)
zero_w = sum(1 for a in train_data["annotations"] if a["bbox"][2]<=0)
zero_h = sum(1 for a in train_data["annotations"] if a["bbox"][3]<=0)
total_issues = neg_x+neg_y+zero_w+zero_h
print(f"  Annotation quality issues (raw): {total_issues} "
      f"(neg_x={neg_x}, neg_y={neg_y}, zero_w={zero_w}, zero_h={zero_h})")
print(f"  → All issues fixed in NB1/NB2 via coordinate clipping to [0,1]")

# ── EDA FIG 1: Class distribution all splits ─────────────────────────
print("\n  Building EDA figures...")
fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle("EDA 1 — Class Distribution (Train / Val / Test)", fontweight="bold")
for ax, (sname, d) in zip(axes, splits.items()):
    cnts = Counter(cat_map[a["category_id"]] for a in d["annotations"])
    bars = ax.bar(cnts.keys(), cnts.values(),
                  color=[COLORS.get(k,"gray") for k in cnts.keys()],
                  edgecolor="white", linewidth=1.2)
    ax.set_title(f"{sname}  (n={len(d['images'])} imgs)")
    ax.set_ylabel("Count")
    for bar, v in zip(bars, cnts.values()):
        ax.text(bar.get_x()+bar.get_width()/2, v+max(cnts.values())*0.01,
                f"{v:,}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_01_class_distribution.png")

# ── EDA FIG 2: Object size + scale pie + aspect ratio ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle("EDA 2 — Object Size, Scale & Aspect Ratio (Train)", fontweight="bold")
axes[0].hist(sizes, bins=50, color="#9C27B0", alpha=0.8, edgecolor="white")
axes[0].axvline(32**2, color="red",    ls="--", lw=1.5, label=f"Small {100*small/total:.0f}%")
axes[0].axvline(96**2, color="orange", ls="--", lw=1.5, label=f"Med {100*medium/total:.0f}%")
axes[0].set_title("BBox Area (px²)"); axes[0].set_xlabel("Area"); axes[0].legend(fontsize=8)
axes[1].pie([small,medium,large],
            labels=[
    f"Small\n{100*small/total:.1f}%",
    f"Medium\n{100*medium/total:.1f}%",
    f"Large\n{100*large/total:.1f}%"
],
            colors=["#F44336","#FF9800","#4CAF50"],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Object Scale Breakdown")
axes[2].hist(ratios, bins=40, color="#00BCD4", alpha=0.8, edgecolor="white")
axes[2].axvline(1.0, color="red", ls="--", label="Square (1:1)")
axes[2].set_title(
    f"Aspect Ratio W/H\nmean={np.mean(ratios):.2f}  std={np.std(ratios):.2f}"
)
axes[2].legend(fontsize=8)
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_02_size_scale_ratio.png")

# ── EDA FIG 3: Density per image + per-class boxplot ─────────────────
class_density = {}
for cn in CLASS_NAMES:
    cid = [k for k,v in cat_map.items() if v==cn][0]
    per = Counter(a["image_id"] for a in train_data["annotations"] if a["category_id"]==cid)
    class_density[cn] = list(per.values())
fig, axes = plt.subplots(1, 2, figsize=(12,4))
fig.suptitle("EDA 3 — Object Density per Image (Train)", fontweight="bold")
axes[0].hist(density, bins=35, color="#F44336", alpha=0.8, edgecolor="white")
axes[0].set_title(f"Objects/Image  mean={np.mean(density):.1f}  max={max(density)}  std={np.std(density):.1f}")
axes[0].set_xlabel("Count")
bp = axes[1].boxplot([class_density[c] for c in CLASS_NAMES],
                     labels=CLASS_NAMES, patch_artist=True)
for patch, color in zip(bp["boxes"],["#2196F3","#FF9800","#4CAF50"]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title("Per-Class Objects/Image"); axes[1].set_ylabel("Count")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_03_density.png")

# ── EDA FIG 4: Spatial heatmap (float16 — memory safe) ───────────────
W_img = int(np.mean(widths)); H_img = int(np.mean(heights))
hmap_all = np.zeros((H_img,W_img), dtype=np.float16)
hmaps    = {cls: np.zeros((H_img,W_img), dtype=np.float16) for cls in CLASS_NAMES}
for ann in train_data["annotations"]:
    x,y,w,h = ann["bbox"]
    x1,y1 = max(0,int(x)),         max(0,int(y))
    x2,y2 = min(W_img-1,int(x+w)), min(H_img-1,int(y+h))
    if x2>x1 and y2>y1:
        hmap_all[y1:y2,x1:x2] += 1
        hmaps[cat_map[ann["category_id"]]][y1:y2,x1:x2] += 1
fig, axes = plt.subplots(1,4,figsize=(18,4))
fig.suptitle("EDA 4 — BBox Spatial Heatmap: Where Objects Appear (Train)", fontweight="bold")
for ax,(title,hm) in zip(axes,[("All Classes",hmap_all)]+[(c,hmaps[c]) for c in CLASS_NAMES]):
    im = ax.imshow(np.array(hm,dtype=np.float32),cmap="hot",interpolation="nearest",aspect="auto")
    ax.set_title(title); ax.axis("off"); plt.colorbar(im,ax=ax,fraction=0.046)
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_04_spatial_heatmap.png")
del hmap_all, hmaps; gc.collect()

# ── EDA FIG 5: Co-occurrence matrix ──────────────────────────────────
img_cls_map = {}
for a in train_data["annotations"]:
    img_cls_map.setdefault(a["image_id"],set()).add(cat_map[a["category_id"]])
cooccur = np.zeros((len(CLASS_NAMES),len(CLASS_NAMES)),dtype=int)
for cs in img_cls_map.values():
    for ai,na in enumerate(CLASS_NAMES):
        for bi,nb_ in enumerate(CLASS_NAMES):
            if na in cs and nb_ in cs: cooccur[ai][bi] += 1
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(pd.DataFrame(cooccur,index=CLASS_NAMES,columns=CLASS_NAMES),
            annot=True, fmt="d", cmap="Blues", linewidths=0.5, ax=ax)
ax.set_title("EDA 5 — Class Co-occurrence (images containing both classes)", fontweight="bold")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_05_cooccurrence.png")

# ── EDA FIG 6: Box W/H scatter per class ─────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle("EDA 6 — BBox Width vs Height per Class (Train)", fontweight="bold")
for ax, cls_name in zip(axes, CLASS_NAMES):
    cid = [k for k,v in cat_map.items() if v==cls_name][0]
    bbs = [a["bbox"] for a in train_data["annotations"] if a["category_id"]==cid]
    ws  = [b[2] for b in bbs]
    hs  = [b[3] for b in bbs]

    ax.scatter(ws, hs, alpha=0.15, s=4, color=COLORS.get(cls_name, "gray"))

    ax.set_title(f"{cls_name}  (n={len(bbs):,})")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.plot([0,max(ws+[1])],[0,max(ws+[1])],"r--",lw=1,alpha=0.5,label="Square")
    ax.legend(fontsize=8)
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_06_bbox_scatter.png")

# ── EDA FIG 7: Stacked class balance across splits ───────────────────
fig, ax = plt.subplots(figsize=(8,5))
split_names = ["Train","Val","Test"]
cnts_all = [
    [Counter(cat_map[a["category_id"]] for a in d["annotations"]).get(cn,0)
     for d in [train_data,val_data,test_data]]
    for cn in CLASS_NAMES
]
x = np.arange(len(split_names)); w = 0.25

for i, (cn, vals) in enumerate(zip(CLASS_NAMES, cnts_all)):
    # Use .get() with a fallback color to prevent IndexError
    c = COLORS.get(cn, "gray")  
    bars = ax.bar(x + (i-1)*w, vals, w, label=cn,
                  color=c, alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+50, f"{v:,}",
                ha="center", fontsize=8, fontweight="bold")

# ── EDA FIG 8: Annotated sample grid — TRAIN (2×3) ───────────────────
ann_by_img = {}
for a in train_data["annotations"]: ann_by_img.setdefault(a["image_id"],[]).append(a)
for split_name, img_dir, split_data in [
    ("Train", TRAIN_IMG, train_data),
    ("Val",   VAL_IMG,   val_data),
    ("Test",  TEST_IMG,  test_data),
]:
    sample_imgs = random.sample(split_data["images"], min(6, len(split_data["images"])))
    ann_s = {}
    for a in split_data["annotations"]: ann_s.setdefault(a["image_id"],[]).append(a)
    fig, axes_g = plt.subplots(2, 3, figsize=(15,10))
    fig.suptitle(f"EDA — {split_name} Split: Sample Images with Ground-Truth Annotations",
                 fontweight="bold")
    for i, img_info in enumerate(sample_imgs):
        ax = axes_g.flatten()[i]
        img_path = os.path.join(img_dir, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None:
            ax.text(0.5,0.5,"Not found",ha="center",va="center",transform=ax.transAxes)
            ax.axis("off"); continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        cnts = {c:0 for c in CLASS_NAMES}
        for ann in ann_s.get(img_info["id"],[]):
            x,y,w,h = [int(v) for v in ann["bbox"]]
            cls = cat_map[ann["category_id"]]
            cv2.rectangle(img,(x,y),(x+w,y+h),CV_COLORS.get(cls,(200,200,200)),2)
            cv2.putText(img,cls,(x,max(0,y-6)),cv2.FONT_HERSHEY_SIMPLEX,0.45,
                        CV_COLORS.get(cls,(200,200,200)),2)
            cnts[cls] += 1
        ax.imshow(img)
        ax.set_title(f"P:{cnts['plane']} S:{cnts['ship']} V:{cnts['vehicle']}  "
                     f"{img_info['width']}×{img_info['height']}", fontsize=9)
        ax.axis("off"); del img
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/eda_08_samples_{split_name.lower()}.png", dpi=120)

del ann_by_img; gc.collect()

# ── EDA FIG 9: Cumulative annotation count across splits ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle("EDA 9 — Annotation Distribution Summary", fontweight="bold")
labels = ["Train","Val","Test"]
total_anns  = [len(train_data["annotations"]),len(val_data["annotations"]),len(test_data["annotations"])]
total_imgs  = [len(train_data["images"]),len(val_data["images"]),len(test_data["images"])]
axes[0].pie(total_anns, labels=[f"{l}\n{v:,}" for l,v in zip(labels,total_anns)],
            colors=["#2196F3","#FF9800","#4CAF50"],autopct="%1.1f%%",startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[0].set_title("Annotation Split")
# EDA FIG 9 - Right Side
axes[1].pie(total_imgs, labels=[f"{l}\n{v:,}" for l,v in zip(labels, total_imgs)],
            colors=["#2196F3","#FF9800","#4CAF50"], autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Image Split (Total Files)")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_09_split_pie.png")

# ── EDA FIG 10: BBox size heatmap per class (log scale) ───────────────
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle("EDA 10 — BBox Size Heatmap per Class (log-scale density)", fontweight="bold")
for ax, cls_name in zip(axes, CLASS_NAMES):
    cid = [k for k,v in cat_map.items() if v==cls_name][0]
    bbs = [a["bbox"] for a in train_data["annotations"] if a["category_id"]==cid]
    ws  = [b[2] for b in bbs]; hs = [b[3] for b in bbs]
    if ws and hs:
        h2d, xe, ye = np.histogram2d(ws, hs, bins=30)
        im = ax.imshow(np.log1p(h2d.T), origin="lower", aspect="auto",
                       extent=[xe[0],xe[-1],ye[0],ye[-1]], cmap="YlOrRd")
        plt.colorbar(im, ax=ax, label="log(count+1)")
    ax.set_title(f"{cls_name}  (n={len(bbs):,})")
    ax.set_xlabel("Width (px)"); ax.set_ylabel("Height (px)")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_10_bbox_heatmap.png")

print("\n" + "="*65)
print("EDA COMPLETE — 10 FIGURES SAVED")
print("="*65)
# Print the EDA CSV if available
if os.path.exists(EDA_CSV):
    print("\nEDA Summary from NB1:")
    print(pd.read_csv(EDA_CSV).to_string(index=False))
print("\n📌 Proceed to Cell 3 — What Was Done in NB1 + NB2 (full audit)")


COMPLETE RAW DATASET STATISTICS
Split  Images  Annotations  Empty imgs  Ann/img  plane  ship  vehicle
Train    1305        44909          30    34.41   7067  4222    33620
  Val      54         2032           1    37.63    259   160     1613
 Test      55         1826           1    33.20    188    27     1611

  Image resolution : 640×640  unique combos: 1
  Small (<32²): 30,996(69.0%)  Med: 12,498(27.8%)  Large: 1,415(3.2%)
  Aspect ratio : mean=1.071  std=0.937
  Ann density  : mean=34.41  max=354  std=39.22
  Annotation quality issues (raw): 724 (neg_x=358, neg_y=366, zero_w=0, zero_h=0)
  → All issues fixed in NB1/NB2 via coordinate clipping to [0,1]

  Building EDA figures...
   ✅ eda_01_class_distribution.png
   ✅ eda_02_size_scale_ratio.png
   ✅ eda_03_density.png
   ✅ eda_04_spatial_heatmap.png
   ✅ eda_05_cooccurrence.png
   ✅ eda_06_bbox_scatter.png
   ✅ eda_08_samples_train.png
   ✅ eda_08_samples_val.png
   ✅ eda_08_samples_test.png
   ✅ eda_09_split_pie.png
   ✅ eda_10_bb

## 📋 Cell 3 — Complete Audit: What Was Done in NB1 + NB2

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — FULL AUDIT OF NB1 + NB2
# Every parameter, every model, every decision documented here.
# ══════════════════════════════════════════════════════════════════════

TRACKED_PARAM_KEYS = [
    "epochs","imgsz","batch","patience","warmup_epochs",
    "mosaic","mixup","copy_paste","degrees","translate",
    "scale","flipud","fliplr","hsv_h","hsv_s",
]

print("=" * 70)
print("WHAT WAS DONE — NOTEBOOK 1 (YOLO11n / YOLO11s / YOLO11m)")
print("=" * 70)
print("""
  DATASET
  ───────
  • Source     : COCO-format satellite aerial imagery
  • Classes    : 3 — plane (0), ship (1), vehicle (2)
  • Format     : COCO JSON → converted to YOLO TXT labels
  • Fix applied: All bbox coordinates clipped to [0,1] to remove
                 negative values and out-of-bound annotations
  • data.yaml  : Written with train/val/test paths and class names

  PREPROCESSING
  ─────────────
  • ID_MAP     : {1:0 (plane), 2:1 (ship), 3:2 (vehicle)}
  • Degenerate boxes (w ≤ 1e-4 or h ≤ 1e-4) removed
  • Images copied from COCO source dirs into YOLO structure

  MODELS TRAINED
  ──────────────
  Model       Weights         Batch   Architecture
  ─────────── ─────────────── ─────── ────────────────────────────────
  YOLO11n     yolo11n.pt      32      Nano — lightest, fastest
  YOLO11s     yolo11s.pt      16      Small — balanced
  YOLO11m     yolo11m.pt       4      Medium — most accurate of YOLO11
  
  Architecture features of YOLO11:
  • C3k2 block (efficient CSP bottleneck)
  • Enhanced spatial attention
  • Anchor-free detection head
  • Same backbone improvements over YOLOv8
""")

print("=" * 70)
print("THE 15 CANONICAL TRAINING HYPERPARAMETERS (NB1 + NB2 IDENTICAL)")
print("=" * 70)
descriptions = {
    "epochs":        "Total training epochs",
    "imgsz":         "Input image resolution (pixels)",
    "batch":         "Mini-batch size (varies per model)",
    "patience":      "Early-stop: halt if no improvement for N epochs",
    "warmup_epochs": "Gradual LR warm-up for first N epochs",
    "mosaic":        "Mosaic aug: combine 4 images into 1 (prob)",
    "mixup":         "MixUp aug: blend two images (prob)",
    "copy_paste":    "Copy-paste aug: paste objects across images (prob)",
    "degrees":       "Random rotation range ±degrees (aerial = 45°)",
    "translate":     "Random translation fraction of image size",
    "scale":         "Random scale jitter fraction",
    "flipud":        "Vertical (up-down) flip probability",
    "fliplr":        "Horizontal (left-right) flip probability",
    "hsv_h":         "Hue colour jitter strength",
    "hsv_s":         "Saturation colour jitter strength",
}
values = {
    "epochs":50,"imgsz":640,"batch":"n=32/s=16/m=4/rt=4/v8=4",
    "patience":20,"warmup_epochs":3,"mosaic":1.0,"mixup":0.1,
    "copy_paste":0.1,"degrees":45.0,"translate":0.1,"scale":0.5,
    "flipud":0.5,"fliplr":0.5,"hsv_h":0.015,"hsv_s":0.7,
}
print(f"  {'ID':<5} {'Parameter':<18} {'Value':<30} Description")
print("  " + "─"*72)
for i,k in enumerate(TRACKED_PARAM_KEYS,1):
    print(f"  P{i:02d}   {k:<18} {str(values[k]):<30} {descriptions[k]}")

print(f"""
  FIXED PARAMS (same for all models, not in tracked 15):
    workers = 2  |  seed = 42  |  verbose = False

  SAHI CONFIGURATION (3 slice sizes, identical for all 5 models):
    SP01  sahi_slice_sizes = [480, 320, 160]  px
    SP02  sahi_overlap     = 0.2
    SP03  sahi_conf        = 0.25
    SP04  sahi_n_imgs      = 50  (images evaluated per ablation)

  OUTPUTS SAVED PER MODEL (NB1):
    • {{model}}_best.pt            — best weights by val mAP
    • runs/{{model}}/weights/      — last.pt + best.pt
    • runs/{{model}}/results.csv   — per-epoch training curves
    • runs/{{model}}/confusion_matrix.png
    • runs/{{model}}/PR_curve.png, F1_curve.png, P_curve.png, R_curve.png
    • saved_models/nb1_results.csv — 9 rows (3 models × 3 SAHI slices)
""")

print("=" * 70)
print("WHAT WAS DONE — NOTEBOOK 2 (RT-DETR-L + YOLOv8m)")
print("=" * 70)
print("""
  MODELS TRAINED
  ──────────────
  Model       Weights         Batch   Architecture
  ─────────── ─────────────── ─────── ────────────────────────────────
  RT-DETR-L   rtdetr-l.pt      4      Transformer (DETReg-style)
                                       — strong on large/medium objects
                                       — slower than YOLO, more global context
  YOLOv8m     yolov8m.pt       4      CNN anchor-free (YOLOv8)
                                       — strong baseline comparison

  SAME 15 hyperparameters as NB1 — verified identical
  SAME SAHI config: [480, 320, 160]px slices

  OUTPUTS SAVED (NB2):
    • RT-DETR-L_best.pt, YOLOv8m_best.pt
    • runs/RT-DETR-L/, runs/YOLOv8m/  (curves, confusion matrix, etc.)
    • saved_models/nb2_results.csv — 6 rows (2 models × 3 SAHI slices)
""")

print("=" * 70)
print("COMBINED RESULTS LOADED FROM NB1 + NB2")
print("=" * 70)
if not df.empty:
    show = ["Model","mAP50","mAP50_95","Precision","Recall","F1",
            "FPS","Params_M","Train_time_hr","std_avg_det","sahi_avg_det","sahi_gain_pct","sahi_slice"]
    show = [c for c in show if c in df.columns]
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 200)
    print(df[show].to_string(index=False))

    # Per-class AP
    print("\n  PER-CLASS AP@50:")
    if all(c in df.columns for c in ["AP50_plane","AP50_ship","AP50_vehicle"]):
        print(f"  {'Model':<20} {'AP50_plane':>12} {'AP50_ship':>12} {'AP50_vehicle':>14}")
        print("  " + "─"*60)
        for _, row in df.drop_duplicates("base_model").iterrows():
            print(f"  {row['base_model']:<20} {row.get('AP50_plane',0):>12.4f} "
                  f"{row.get('AP50_ship',0):>12.4f} {row.get('AP50_vehicle',0):>14.4f}")

# 15-param consistency check
print("\n" + "="*70)
print("15-PARAMETER CONSISTENCY CHECK (NB1 vs NB2)")
print("="*70)
hp_cols = [f"hp_{k}" for k in TRACKED_PARAM_KEYS if f"hp_{k}" in df.columns]
if hp_cols:
    issues = []
    for k in TRACKED_PARAM_KEYS:
        col = f"hp_{k}"
        if col not in df.columns or k=="batch": continue
        vals = df[col].astype(str).unique()
        if len(vals) > 1: issues.append((k, vals.tolist()))
    if issues:
        print("  ⚠️  INCONSISTENCIES FOUND:")
        for k,v in issues: print(f"     {k}: {v}")
    else:
        print("  ✅ All 14 non-batch params are IDENTICAL across NB1 and NB2.")
        print("     batch differs by design: n=32, s=16, m=4, rtdetr=4, v8m=4")
else:
    print("  ⚠️  hp_* columns not found — CSV may be from older NB version.")

print("\n📌 Proceed to Cell 4 — Model Predictions on All Splits (before/after SAHI)")


WHAT WAS DONE — NOTEBOOK 1 (YOLO11n / YOLO11s / YOLO11m)

  DATASET
  ───────
  • Source     : COCO-format satellite aerial imagery
  • Classes    : 3 — plane (0), ship (1), vehicle (2)
  • Format     : COCO JSON → converted to YOLO TXT labels
  • Fix applied: All bbox coordinates clipped to [0,1] to remove
                 negative values and out-of-bound annotations
  • data.yaml  : Written with train/val/test paths and class names

  PREPROCESSING
  ─────────────
  • ID_MAP     : {1:0 (plane), 2:1 (ship), 3:2 (vehicle)}
  • Degenerate boxes (w ≤ 1e-4 or h ≤ 1e-4) removed
  • Images copied from COCO source dirs into YOLO structure

  MODELS TRAINED
  ──────────────
  Model       Weights         Batch   Architecture
  ─────────── ─────────────── ─────── ────────────────────────────────
  YOLO11n     yolo11n.pt      32      Nano — lightest, fastest
  YOLO11s     yolo11s.pt      16      Small — balanced
  YOLO11m     yolo11m.pt       4      Medium — most accurate of YOLO11
  
  Architec

## 🖼️ Cell 4 — Model Predictions on Train / Val / Test (Standard + SAHI)
> Loads all 5 models and runs predictions on sample images from each split.  
> Shows standard inference vs SAHI-sliced side by side.

In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — PREDICTIONS ON ALL SPLITS: STANDARD vs SAHI
# ══════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import gc, os, cv2, random
import numpy as np
import matplotlib.pyplot as plt

CONF       = 0.25
SAHI_BEST  = 320    # use 320px slice for the "after SAHI" visualisation
SAHI_OV    = 0.2
N_SAMPLE   = 3      # images per split per model

# ── Load models ──────────────────────────────────────────────────────
models_loaded = {}
for name, pt in MODEL_PATHS.items():
    if os.path.exists(pt):
        models_loaded[name] = YOLO(pt)
        print(f"  ✅ {name} loaded")
    else:
        print(f"  ⚠️  {name} not found: {pt}")

# ── Helper: draw predictions on image ────────────────────────────────
def predict_draw(model, img_path, conf=CONF):
    res = model.predict(img_path, conf=conf, verbose=False, device=device)
    if res and len(res) > 0:
        return res[0].plot()[:,:,::-1]   # BGR→RGB
    img = cv2.imread(img_path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

def sahi_draw(model_path, img_path, slice_size=SAHI_BEST, conf=CONF):
    det = AutoDetectionModel.from_pretrained(
        model_type="yolov8", model_path=model_path,
        confidence_threshold=conf, device=device)
    r = get_sliced_prediction(
        img_path, det,
        slice_height=slice_size, slice_width=slice_size,
        overlap_height_ratio=SAHI_OV, overlap_width_ratio=SAHI_OV,
        verbose=0)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for pred in r.object_prediction_list:
        bb = pred.bbox
        x1,y1,x2,y2 = int(bb.minx),int(bb.miny),int(bb.maxx),int(bb.maxy)
        cls = pred.category.name
        color = CV_COLORS.get(cls,(200,200,200))
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
        cv2.putText(img,f"{cls} {pred.score.value:.2f}",
                    (x1,max(0,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,0.4,color,1)
    del det; gc.collect()
    return img

# ── For each split: show 3 images × best model (standard + SAHI) ─────
for split_name, img_dir, split_data in [
    ("Train", TRAIN_IMG, train_data),
    ("Val",   VAL_IMG,   val_data),
    ("Test",  TEST_IMG,  test_data),
]:
    # pick model with highest mAP50 (best slice)
    if not df.empty:
        df_best_row = (df.sort_values("sahi_gain_pct",ascending=False)
                         .drop_duplicates("base_model")
                         .sort_values("mAP50",ascending=False)
                         .iloc[0])
        best_model_name = df_best_row["base_model"]
    else:
        best_model_name = list(models_loaded.keys())[0] if models_loaded else None

    if best_model_name not in models_loaded:
        print(f"  ⚠️  {split_name}: best model {best_model_name} not loaded, skipping.")
        continue

    model_obj = models_loaded[best_model_name]
    pt_path   = MODEL_PATHS[best_model_name]

    sample_imgs = random.sample(split_data["images"], min(N_SAMPLE, len(split_data["images"])))
    fig, axes = plt.subplots(N_SAMPLE, 2, figsize=(14, N_SAMPLE*5))
    fig.suptitle(f"Predictions on {split_name} Split — {best_model_name}\nLeft: Standard  |  Right: SAHI (slice={SAHI_BEST}px)",
                 fontsize=13, fontweight="bold")
    if N_SAMPLE == 1: axes = [axes]

    for row_i, img_info in enumerate(sample_imgs):
        img_path = os.path.join(img_dir, img_info["file_name"])
        if not os.path.exists(img_path):
            for ax in axes[row_i]: ax.text(0.5,0.5,"Not found",ha="center",va="center",transform=ax.transAxes); ax.axis("off")
            continue
        std_img  = predict_draw(model_obj, img_path)
        sahi_img = sahi_draw(pt_path, img_path)
        for ax, img, title in zip(axes[row_i],
                                   [std_img, sahi_img],
                                   ["Standard inference", f"SAHI slice={SAHI_BEST}px"]):
            if img is not None: ax.imshow(img)
            ax.set_title(title, fontsize=10); ax.axis("off")

    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/pred_{split_name.lower()}_std_vs_sahi.png", dpi=130)

# ── Show predictions from ALL 5 models on the same test image ────────
test_imgs_paths = [os.path.join(TEST_IMG, f["file_name"])
                   for f in test_data["images"]
                   if os.path.exists(os.path.join(TEST_IMG, f["file_name"]))][:1]

if test_imgs_paths and models_loaded:
    img_path = test_imgs_paths[0]
    n_m      = len(models_loaded)
    fig, axes = plt.subplots(1, n_m, figsize=(5*n_m, 5))
    fig.suptitle("All Models on Same Test Image (Standard Inference, conf=0.25)",
                 fontsize=13, fontweight="bold")
    for ax, (mname, m) in zip(axes.flatten() if n_m>1 else [axes], models_loaded.items()):
        r = predict_draw(m, img_path)
        if r is not None: ax.imshow(r)
        ax.set_title(mname, fontsize=10); ax.axis("off")
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/pred_all5_same_image.png", dpi=150)

# ── SAHI 3-slice comparison on one test image (best model) ───────────
if test_imgs_paths and best_model_name in models_loaded:
    img_path = test_imgs_paths[0]
    pt_path  = MODEL_PATHS[best_model_name]
    slices   = [480, 320, 160]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"SAHI 3-Slice Comparison on Test Image — {best_model_name}",
                 fontsize=13, fontweight="bold")
    std_img = predict_draw(models_loaded[best_model_name], img_path)
    axes[0].imshow(std_img if std_img is not None else np.zeros((100,100,3),dtype=np.uint8))
    axes[0].set_title("Standard (no SAHI)"); axes[0].axis("off")
    for ax, s_size in zip(axes[1:], slices):
        s_img = sahi_draw(pt_path, img_path, slice_size=s_size)
        ax.imshow(s_img)
        ax.set_title(f"SAHI slice={s_size}px"); ax.axis("off")
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/pred_sahi_3slice_comparison.png", dpi=150)

print("\n📌 Proceed to Cell 5 — Publication Figures + LaTeX Table")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  ✅ YOLO11n loaded
  ✅ YOLO11s loaded
  ✅ YOLO11m loaded
  ✅ RT-DETR-L loaded
  ✅ YOLOv8m loaded
   ✅ pred_train_std_vs_sahi.png
   ✅ pred_val_std_vs_sahi.png
   ✅ pred_test_std_vs_sahi.png
   ✅ pred_all5_same_image.png
   ✅ pred_sahi_3slice_comparison.png

📌 Proceed to Cell 5 — Publication Figures + LaTeX Table


## 📊 Cell 5 — 12 Publication Figures (300 DPI) + LaTeX Table

In [8]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — 12 PUBLICATION-READY FIGURES + LATEX TABLE
# ══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

if df.empty:
    print("❌ No results loaded — run Cell 1 first.")
else:
    # Best-slice df (one row per base model)
    df_best = (df.sort_values("sahi_gain_pct",ascending=False)
                 .drop_duplicates("base_model")
                 .sort_values("mAP50",ascending=False)
                 .reset_index(drop=True))
    MODEL_NAMES = df_best["base_model"].tolist()
    N = len(MODEL_NAMES)
    
    # Fallback palette if not defined earlier
    try:
        PALETTE_N = PALETTE[:N]
    except NameError:
        PALETTE_N = sns.color_palette("husl", N)

    # ── FIG P1: 6-metric bar chart ────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18,10))
    fig.suptitle("Fig 1 — Comprehensive Model Comparison (best SAHI slice per model)",
                 fontsize=14, fontweight="bold", y=1.01)
    bar_metrics = [
        ("mAP50","mAP@50",True),("mAP50_95","mAP@50:95",True),
        ("F1","F1 Score",True),("FPS","Inference FPS",True),
        ("Params_M","Parameters (M)",False),("sahi_gain_pct","SAHI Gain %",True),
    ]
    for ax,(col,label,hi) in zip(axes.flat,bar_metrics):
        if col not in df_best.columns:
            ax.text(0.5,0.5,f"N/A",ha="center",va="center",transform=ax.transAxes); continue
        vals = df_best[col].fillna(0).tolist()
        bars = ax.bar(MODEL_NAMES,vals,color=PALETTE_N,edgecolor="white",linewidth=1.2)
        ax.set_title(f"{label}\n({'higher' if hi else 'lower'} is better)")
        ax.set_xticklabels(MODEL_NAMES,rotation=20,ha="right",fontsize=9)
        bi = int(np.argmax(vals)) if hi else int(np.argmin(vals))
        bars[bi].set_edgecolor("gold"); bars[bi].set_linewidth(3)
        for bar,v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.002*max(max(vals),1e-6),
                    f"{v:.3f}",ha="center",va="bottom",fontsize=8,fontweight="bold")
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/fig1_main_comparison.png", dpi=300)

    # ── FIG P2: Radar chart ───────────────────────────────────────────
    radar_m = ["mAP50","mAP50_95","Precision","Recall","F1"]
    angles  = np.linspace(0,2*np.pi,len(radar_m),endpoint=False).tolist()+[0]
    fig, ax = plt.subplots(figsize=(8,8),subplot_kw=dict(polar=True))
    for i,(_,row) in enumerate(df_best.iterrows()):
        vals = [row.get(m,0)/max(df_best[m].max(),1e-6) for m in radar_m]+[row.get(radar_m[0],0)/max(df_best[radar_m[0]].max(),1e-6)]
        ax.plot(angles,vals,"o-",lw=2,color=PALETTE_N[i],label=row["base_model"])
        ax.fill(angles,vals,alpha=0.1,color=PALETTE_N[i])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_m,fontsize=11)
    ax.set_title("Fig 2 — Performance Radar (normalised per metric)",fontsize=13,fontweight="bold",pad=20)
    ax.legend(loc="upper right",bbox_to_anchor=(1.35,1.15))
    save_close(fig, f"{PLOT_DIR}/fig2_radar.png", dpi=300)

    # ── FIG P3: Per-class AP heatmap ──────────────────────────────────
    cls_cols = ["AP50_plane","AP50_ship","AP50_vehicle"]
    if all(c in df_best.columns for c in cls_cols):
        matrix = df_best.set_index("base_model")[cls_cols].rename(
            columns={"AP50_plane":"Plane","AP50_ship":"Ship","AP50_vehicle":"Vehicle"})
        fig, ax = plt.subplots(figsize=(7,5))
        sns.heatmap(matrix.astype(float),annot=True,fmt=".4f",cmap="YlOrRd",
                    linewidths=0.5,ax=ax,vmin=0,vmax=1,cbar_kws={"label":"AP@50"})
        ax.set_title("Fig 3 — Per-Class AP@50 Heatmap",fontweight="bold")
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig3_per_class_ap.png", dpi=300)

    # ── FIG P4: 3-Slice SAHI ablation ─────────────────────────────────
    fig, axes = plt.subplots(1,2,figsize=(16,6))
    fig.suptitle("Fig 4 — 3-Slice SAHI Ablation: Standard vs Tiled Inference",
                 fontsize=14,fontweight="bold")
    slice_colors = {480:"#1976D2",320:"#FF5722",160:"#4CAF50"}
    x = np.arange(N); w = 0.2
    for ax_i,(ax,metric,ylabel) in enumerate(zip(axes,
        ["sahi_avg_det","sahi_gain_pct"],
        ["Avg Detections per Image","SAHI Gain % over Standard"])):
        for si,s_size in enumerate(sorted(SAHI_SIZES_R,reverse=True)):
            df_s = df[df["sahi_slice"]==s_size].set_index("base_model").reindex(MODEL_NAMES)
            vals = df_s[metric].fillna(0).tolist() if metric in df_s.columns else [0]*N
            offset = (si - len(SAHI_SIZES_R)/2 + 0.5)*w
            bars = ax.bar(x+offset,vals,w,label=f"Slice {int(s_size)}px",
                          color=slice_colors.get(s_size,"gray"),alpha=0.85)
            for bar,v in zip(bars,vals):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.03,
                        f"{v:.1f}",ha="center",va="bottom",fontsize=7)
        if ax_i==0:
            std_means = [df[df["base_model"]==m]["std_avg_det"].mean() for m in MODEL_NAMES]
            ax.plot(x,std_means,"k--o",lw=1.5,ms=5,label="Standard (no SAHI)",zorder=5)
        ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES,rotation=15,fontsize=9)
        ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=8)
        if ax_i==1: ax.axhline(0,color="black",lw=1)
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/fig4_sahi_ablation_3slice.png", dpi=300)

    # ── FIG P5: Efficiency scatter ─────────────────────────────────────
    if "Params_M" in df_best.columns:
        fig, ax = plt.subplots(figsize=(9,6))
        for i,(_,row) in enumerate(df_best.iterrows()):
            ax.scatter(row["Params_M"],row.get("mAP50",0),
                       s=max(row.get("FPS",10),5)*5,c=PALETTE_N[i],
                       alpha=0.85,edgecolors="white",linewidth=1.5,zorder=3)
            ax.annotate(row["base_model"],(row["Params_M"],row.get("mAP50",0)),
                        textcoords="offset points",xytext=(8,4),fontsize=9)
        ax.set_xlabel("Parameters (M)  ← smaller is lighter")
        ax.set_ylabel("mAP@50  → higher is better")
        ax.set_title("Fig 5 — Accuracy vs Model Size (bubble size = FPS)",fontweight="bold")
        for fps_val in [30,60,120]:
            ax.scatter([],[],s=fps_val*5,c="gray",alpha=0.5,label=f"{fps_val} FPS")
        ax.legend(title="FPS",loc="lower right")
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig5_efficiency.png", dpi=300)

    # ── FIG P6: mAP50 vs mAP50:95 grouped ─────────────────────────────
    if "mAP50_95" in df_best.columns:
        x = np.arange(N); w2=0.35
        fig, ax = plt.subplots(figsize=(11,6))
        ax.bar(x-w2/2,df_best["mAP50"].fillna(0),w2,label="mAP@50",color="#2196F3",alpha=0.85)
        ax.bar(x+w2/2,df_best["mAP50_95"].fillna(0),w2,label="mAP@50:95",color="#FF9800",alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES,rotation=15)
        ax.set_ylabel("mAP Score"); ax.set_ylim(0,1.05)
        ax.set_title("Fig 6 — mAP@50 vs mAP@50:95",fontweight="bold"); ax.legend()
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig6_map_comparison.png", dpi=300)

    # ── FIG P7: Training time vs accuracy ─────────────────────────────
    if "Train_time_hr" in df_best.columns:
        fig, ax = plt.subplots(figsize=(9,6))
        for i,(_,row) in enumerate(df_best.iterrows()):
            ax.scatter(row.get("Train_time_hr",0),row.get("mAP50",0),
                       s=120,c=PALETTE_N[i],alpha=0.9,edgecolors="white",lw=1.5,zorder=3)
            ax.annotate(row["base_model"],(row.get("Train_time_hr",0),row.get("mAP50",0)),
                        textcoords="offset points",xytext=(8,4),fontsize=9)
        ax.set_xlabel("Training Time (hours)")
        ax.set_ylabel("mAP@50")
        ax.set_title("Fig 7 — Training Time vs Accuracy",fontweight="bold")
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig7_traintime_vs_accuracy.png", dpi=300)

    # ── FIG P8: Precision-Recall tradeoff ─────────────────────────────
    if "Precision" in df_best.columns and "Recall" in df_best.columns:
        fig, ax = plt.subplots(figsize=(8,6))
        for i,(_,row) in enumerate(df_best.iterrows()):
            ax.scatter(row.get("Recall",0),row.get("Precision",0),
                       s=120,c=PALETTE_N[i],alpha=0.9,edgecolors="white",lw=1.5,zorder=3,
                       label=row["base_model"])
            ax.annotate(row["base_model"],(row.get("Recall",0),row.get("Precision",0)),
                        textcoords="offset points",xytext=(6,4),fontsize=9)
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.set_xlim(0,1.05); ax.set_ylim(0,1.05)
        ax.set_title("Fig 8 — Precision vs Recall Tradeoff",fontweight="bold")
        ax.legend(fontsize=9)
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig8_precision_recall.png", dpi=300)

    # ── FIG P9: Stacked per-class AP ──────────────────────────────────
    cls_cols = ["AP50_plane","AP50_ship","AP50_vehicle"]
    if all(c in df_best.columns for c in cls_cols):
        x = np.arange(N); w3=0.2
        fig, ax = plt.subplots(figsize=(12,6))
        for i,(col,cn) in enumerate(zip(cls_cols,["Plane","Ship","Vehicle"])):
            offset = (i-1)*w3
            bars = ax.bar(x+offset,df_best[col].fillna(0),w3,
                          label=cn,color=list(COLORS.values())[i],alpha=0.85)
            for bar,v in zip(bars,df_best[col].fillna(0)):
                ax.text(bar.get_x()+bar.get_width()/2,v+0.005,f"{v:.3f}",
                        ha="center",va="bottom",fontsize=7)
        ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES,rotation=15)
        ax.set_ylabel("AP@50"); ax.set_ylim(0,1.05)
        ax.set_title("Fig 9 — Per-Class AP@50 Comparison",fontweight="bold"); ax.legend()
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig9_per_class_ap_bars.png", dpi=300)

    # ── FIG P10: SAHI gain per class ──────────────────────────────────
    sahi_gain_by_slice = []
    for s_size in sorted(SAHI_SIZES_R,reverse=True):
        df_s = df[df["sahi_slice"]==s_size]
        for _,row in df_s.iterrows():
            sahi_gain_by_slice.append({
                "Model":      row.get("base_model",row["Model"]),
                "SAHI_slice": int(s_size),
                "gain":       row.get("sahi_gain_pct",0),
            })
    if sahi_gain_by_slice:
        df_sg = pd.DataFrame(sahi_gain_by_slice)
        fig, ax = plt.subplots(figsize=(12,5))
        for i,s_size in enumerate(sorted(SAHI_SIZES_R,reverse=True)):
            sub = df_sg[df_sg["SAHI_slice"]==int(s_size)]
            ax.plot(sub["Model"],sub["gain"],"o-",
                    label=f"Slice {s_size}px",lw=2,ms=8,
                    color=list(slice_colors.values())[i] if i<len(slice_colors) else "gray")
        ax.axhline(0,color="black",lw=1,ls="--")
        ax.set_ylabel("SAHI Gain % over Standard"); ax.set_xlabel("Model")
        ax.set_title("Fig 10 — SAHI Gain % by Slice Size (all models)",fontweight="bold")
        ax.legend(); plt.xticks(rotation=15)
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig10_sahi_gain_by_slice.png", dpi=300)

    # ── FIG P11: Statistical delta heatmap ────────────────────────────
    sig_rows = []
    baseline_m = "YOLOv8m"
    brow = df_best[df_best["base_model"]==baseline_m]
    for _,row in df_best.iterrows():
        if row["base_model"]==baseline_m: continue
        for m in ["mAP50","mAP50_95","F1","Precision","Recall"]:
            if m not in df_best.columns: continue
            val  = float(row[m]) if not pd.isna(row[m]) else 0
            base = float(brow[m].values[0]) if len(brow)>0 else 0
            sig_rows.append({"Model":row["base_model"],"Metric":m,"Delta":round(val-base,4)})
    if sig_rows:
        df_sig = pd.DataFrame(sig_rows)
        pivot  = df_sig.pivot(index="Model",columns="Metric",values="Delta").fillna(0)
        fig, ax = plt.subplots(figsize=(9,5))
        sns.heatmap(pivot,annot=True,fmt=".4f",cmap="RdYlGn",center=0,
                    linewidths=0.5,ax=ax,cbar_kws={"label":f"Δ vs {baseline_m}"})
        ax.set_title(f"Fig 11 — Δ vs {baseline_m} Baseline (green=better, red=worse)",
                     fontweight="bold")
        plt.tight_layout()
        save_close(fig, f"{PLOT_DIR}/fig11_delta_heatmap.png", dpi=300)
        df_sig.to_csv(f"{WORK_DIR}/statistical_comparison.csv",index=False)

    # ── FIG P12: Full results table as figure ─────────────────────────
    show_cols = ["base_model","mAP50","mAP50_95","F1","Precision","Recall",
                 "AP50_plane","AP50_ship","AP50_vehicle","FPS","Params_M",
                 "Train_time_hr","sahi_slice","std_avg_det","sahi_avg_det","sahi_gain_pct"]
    show_cols = [c for c in show_cols if c in df_best.columns]
    fig, ax = plt.subplots(figsize=(22, max(3, N+2)))
    ax.axis("off")
    col_labels = [c.replace("_"," ") for c in show_cols]
    tbl_data   = [[f"{row[c]:.4f}" if isinstance(row[c],float) else str(row[c])
                   for c in show_cols] for _,row in df_best.iterrows()]
    tbl = ax.table(cellText=tbl_data, colLabels=col_labels,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    tbl.auto_set_column_width(range(len(show_cols)))
    for (r,c),cell in tbl.get_celld().items():
        if r==0: cell.set_facecolor("#2196F3"); cell.set_text_props(color="white",fontweight="bold")
        elif r%2==0: cell.set_facecolor("#F5F5F5")
    ax.set_title("Fig 12 — Complete Results Table (best SAHI slice per model)",
                 fontsize=13, fontweight="bold", pad=20)
    plt.tight_layout()
    save_close(fig, f"{PLOT_DIR}/fig12_results_table.png", dpi=200)

    # ── LaTeX table ───────────────────────────────────────────────────
    latex_cols = ["base_model","mAP50","mAP50_95","F1","Precision","Recall",
                  "AP50_plane","AP50_ship","AP50_vehicle","FPS","Params_M",
                  "sahi_slice","std_avg_det","sahi_avg_det","sahi_gain_pct"]
    latex_cols = [c for c in latex_cols if c in df_best.columns]
    sub        = df_best[latex_cols].copy()
    col_hdr    = ["Model","mAP@50","mAP@50:95","F1","Prec","Rec",
                  "AP-Plane","AP-Ship","AP-Veh","FPS","Params","Slice","Std","SAHI","Gain%"][:len(latex_cols)]
    best_v     = {c: sub[c].max() for c in latex_cols[1:]}
    lines = [
        "\\begin{table*}[t]","\\centering",
        "\\caption{Comparison of five satellite aerial object detectors. "
        "15 hyperparameters identical across all models. "
        "SAHI evaluated at 3 slice sizes (480/320/160px). Best in \\textbf{bold}.}",
        "\\label{tab:results}","\\resizebox{\\textwidth}{!}{%",
        "\\begin{tabular}{l"+"r"*(len(latex_cols)-1)+"}",
        "\\toprule"," & ".join(col_hdr)+" \\\\","\\midrule",
    ]
    for _,row in sub.iterrows():
        vals=[row["base_model"]]
        for c in latex_cols[1:]:
            v = f"{int(row[c])}" if c=="sahi_slice" else f"{float(row[c]):.4f}"
            try:
                if abs(float(row[c])-float(best_v[c]))<1e-4: v=f"\\textbf{{{v}}}"
            except: pass
            vals.append(v)
        lines.append(" & ".join(vals)+" \\\\")
    lines+=["\\bottomrule","\\end{tabular}}","\\end{table*}"]
    with open(f"{WORK_DIR}/table_main_results.tex","w") as f:
        f.write("\n".join(lines))
    print(f"✅ LaTeX table → {WORK_DIR}/table_main_results.tex")

    # ── Final summary ──────────────────────────────────────────────────
    all_figs = sorted(Path(PLOT_DIR).glob("*.png"))
    print("\n" + "="*65)
    print(f"ALL OUTPUTS GENERATED  ({len(all_figs)} figures)")
    print("="*65)
    print("\n  DATASET EDA FIGURES:")
    for fp in all_figs:
        if fp.name.startswith("eda"): print(f"    📊 {fp.name}")
    print("\n  PREDICTION FIGURES:")
    for fp in all_figs:
        if fp.name.startswith("pred"): print(f"    🖼️  {fp.name}")
    print("\n  PUBLICATION FIGURES (300 DPI):")
    for fp in all_figs:
        if fp.name.startswith("fig"): print(f"    📈 {fp.name}")
    print("\n  FILES:")
    print(f"    📄 table_main_results.tex")
    print(f"    📄 statistical_comparison.csv")

   ✅ fig1_main_comparison.png
   ✅ fig2_radar.png
   ✅ fig3_per_class_ap.png
   ✅ fig4_sahi_ablation_3slice.png
   ✅ fig5_efficiency.png
   ✅ fig6_map_comparison.png
   ✅ fig7_traintime_vs_accuracy.png
   ✅ fig8_precision_recall.png
   ✅ fig9_per_class_ap_bars.png
   ✅ fig10_sahi_gain_by_slice.png
   ✅ fig11_delta_heatmap.png
   ✅ fig12_results_table.png
✅ LaTeX table → /kaggle/working/table_main_results.tex

ALL OUTPUTS GENERATED  (28 figures)

  DATASET EDA FIGURES:
    📊 eda_01_class_distribution.png
    📊 eda_02_size_scale_ratio.png
    📊 eda_03_density.png
    📊 eda_04_spatial_heatmap.png
    📊 eda_05_cooccurrence.png
    📊 eda_06_bbox_scatter.png
    📊 eda_08_samples_test.png
    📊 eda_08_samples_train.png
    📊 eda_08_samples_val.png
    📊 eda_09_split_pie.png
    📊 eda_10_bbox_heatmap.png

  PREDICTION FIGURES:
    🖼️  pred_all5_same_image.png
    🖼️  pred_sahi_3slice_comparison.png
    🖼️  pred_test_std_vs_sahi.png
    🖼️  pred_train_std_vs_sahi.png
    🖼️  pred_val_std_vs_sah